# Natural Language to SAN: Embedding Pipeline Demo

## Motivation
While traditional embeddings and cosine similarity aren't likely the most definitively successful method compared to full SOTA LLMs, this approach still serves as an incredibly valuable testbed and baseline for us. It performs much more efficiently than current SLMs, and computation-wise, it is significantly smaller and more lightweight than hitting SOTA LLM APIs like Claude or GPT-4. This makes the embedding method a very strong efficiency-to-performance tradeoff to analyze in the pipeline.

This notebook showcases a self-contained example of the Natural Language to Standard Algebraic Notation (SAN) embedding pipeline.

In [3]:
import chess
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1089.05it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def generate_templates(board, move, san) -> list[str]:
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)
    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None
    templates = []
    
    file_name = chess.FILE_NAMES[chess.square_file(move.from_square)]

    if is_castle:
        if chess.square_file(move.to_square) == 6:
            templates += ["Castle kingside", "Kingside castle", "Perform kingside castling", "Short castle"]
        else:
            templates += ["Castle queenside", "Queenside castle", "Perform queenside castling", "Long castle"]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [f"Promote pawn to {promo_piece} on {target}", f"Advance pawn to {target} and promote to {promo_piece}"]
        return templates

    p_name = chess.piece_name(piece.piece_type)
    if is_capture:
        templates += [f"{p_name.capitalize()} captures on {target}", f"Take on {target} using the {p_name}"]
    elif piece.piece_type == chess.PAWN:
        templates += [f"Advance pawn to {target}", f"Push the pawn to {target}", f"Advance {file_name} pawn"]
    else:
        templates += [f"Move the {p_name} to {target}", f"Play {p_name} to {target}", f"Develop the {p_name} to {target}"]
    
    templates.append(f"Play {san}")
    return templates

def process_board_state(board: chess.Board, user_query: str) -> str:
    records = []
    for move in board.legal_moves:
        san = board.san(move)
        for desc in generate_templates(board, move, san):
            records.append({"move": san, "desc": desc})
            
    descriptions = [r["desc"] for r in records]
    embeddings = embed_model.encode(descriptions, convert_to_numpy=True)
    
    query_vec = embed_model.encode(user_query, convert_to_numpy=True)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = embeddings @ query_vec / (norms + 1e-9)
    best_idx = int(np.argmax(scores))
    
    return records[best_idx]["move"]

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {LLM_MODEL_ID} tokenizer + model (this may take a minute or two)...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"{LLM_MODEL_ID} loaded ✓")

def qwen_pick_move(user_query: str, board: chess.Board) -> str:
    legal_sans = sorted([board.san(m) for m in board.legal_moves])
    fen = board.fen()

    prompt = (
        f"You are a chess assistant. Given the board position (FEN) and a list of legal moves, "
        f"pick the single best move that matches the user's intent.\n\n"
        f"FEN: {fen}\n"
        f"Legal moves: {', '.join(legal_sans)}\n"
        f"User says: \"{user_query}\"\n\n"
        f"Reply with ONLY the move in standard algebraic notation (SAN), nothing else."
    )

    messages = [{"role": "user", "content": prompt}]
    input_text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(input_text, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    cleaned = raw.split()[0] if raw else ""
    if cleaned in legal_sans:
        return cleaned
    
    for san in legal_sans:
        if san in raw:
            return san
        
    return cleaned

Loading Qwen/Qwen2.5-1.5B-Instruct tokenizer + model (this may take a minute or two)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:07<00:00, 45.04it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Qwen/Qwen2.5-1.5B-Instruct loaded ✓


In [ ]:
# --- Comparative Benchmarks: Embedded vs SLM ---

# Small set of varied queries checking the performance of Embeddings vs Qwen
# We use tests 3, 6, and 9 from our larger benchmark evaluation to demonstrate.

demo_test_cases = [
    # (Move Sequence, Natural Language Query, Expected SAN)
    ([], "develop the queen's knight", "Nc3"),
    (["e4", "e5"], "develop the bishop to c4", "Bc4"),
    (["e4", "e5", "Nf3", "Nc6", "Bb5", "a6"], "retreat the bishop to a4", "Ba4"),
]

print("=" * 75)
print(f"{'Query':<35} | {'MiniLM Embeds':<16} | {'Qwen 1.5B (SLM)':<16}")
print("=" * 75)

for move_seq, query, expected in demo_test_cases:
    # Setup board for this specific test case
    test_board = chess.Board()
    for move in move_seq:
        test_board.push_san(move)
        
    # 1. Pipeline execution through embeddings
    embed_result = process_board_state(test_board, query)
    
    # 2. Pipeline execution through SLM
    qwen_result = qwen_pick_move(query, test_board)
    
    # Simple display logic
    embed_display = f"{embed_result} {'(✓)' if embed_result == expected else '(✗)'}"
    qwen_display = f"{qwen_result} {'(✓)' if qwen_result == expected else '(✗)'}"
    
    print(f"{query:<35} | {embed_display:<16} | {qwen_display:<16}")

print("=" * 75)
print("\nThe embeddings execute almost instantly entirely locally,")
print("while the generative SLM uses an order of magnitude more execution compute / memory to perform a similar mapping.")

Query                               | MiniLM Embeds    | Qwen 1.5B (SLM) 
